### Randomized Controlled Trial (RCT)

A **Randomized Controlled Trial (RCT)** is an experiment where you randomly assign subjects to either:

- **Treatment group:** Receives the intervention
- **Control group:** Does not receive the intervention

> **The Golden Rule:** Randomization breaks all back-door paths. If you truly randomize, **correlation = causation**.

Why RCTs Are the Gold Standard?

| Problem | Observational Data | RCT |
|---------|-------------------|-----|
| **Confounding** | X ← Z → Y ❌ | Z is balanced ✅ |
| **Causality** | Correlation ≠ Causation | Correlation = Causation |

X ← Z → Y  (Confounding exists)

X → Y  (No confounders because Z is balanced across groups)

> When you randomize, you ensure that on average, the treatment and control groups are identical in every way except for the treatment itself.


**Randomization = No Confounders!** 🎯


RCT Checklist:

| Step | Action |
|------|--------|
| **1. Hypothesis** | "Treatment X will increase Outcome Y by Z%" |
| **2. Sample Size** | Calculate using power analysis |
| **3. Randomization** | Use proper randomization (not alternating, not time-based) |
| **4. Run Experiment** | Collect data without peeking |
| **5. Analyze** | Difference in means + Confidence intervals |
| **6. Interpret** | If p < 0.05, launch. If not, learn. |

In [2]:
import numpy as np
import pandas as pd
from scipy.stats import ttest_ind, norm
import matplotlib.pyplot as plt

In [3]:
# Scenario: Does a new website design (Treatment) increase conversion rate?
# Parameters
baseline_ctr = 0.05  # 5% conversion rate (control)
expected_lift = 0.01  # Expected lift to 6% conversion
alpha = 0.05         # Significance level (Type I error)
power = 0.80         # Power (1 - Type II error)

# Sample size calculation (simplified)
def calculate_sample_size(baseline, lift, alpha=0.05, power=0.80):
    """
    Calculate minimum sample size needed for an A/B test.
    """
    # Effect size (Cohen's h for proportions)
    p1 = baseline
    p2 = baseline + lift
    effect_size = 2 * np.arcsin(np.sqrt(p2)) - 2 * np.arcsin(np.sqrt(p1))
    
    # Z-scores
    z_alpha = norm.ppf(1 - alpha/2)  # Two-tailed
    z_beta = norm.ppf(power)
    
    # Sample size formula
    n = (2 * (z_alpha + z_beta)**2) / (effect_size**2)
    return int(np.ceil(n))

n_per_group = calculate_sample_size(baseline_ctr, expected_lift)
print(f"Required sample size per group: {n_per_group:,}")
print(f"Total sample size: {n_per_group * 2:,}\n")

# Simulate the experiment
np.random.seed(42)
n = n_per_group

# Generate conversions (Bernoulli trials)
control = np.random.binomial(1, baseline_ctr, n)
treatment = np.random.binomial(1, baseline_ctr + expected_lift, n)

# Analyze results
control_ctr = control.mean()
treatment_ctr = treatment.mean()
lift = treatment_ctr - control_ctr

# Statistical test
t_stat, p_value = ttest_ind(treatment, control)

print("A/B TEST RESULTS:")
print(f"Control CTR: {control_ctr:.2%}")
print(f"Treatment CTR: {treatment_ctr:.2%}")
print(f"Lift: {lift:.2%}")
print(f"P-value: {p_value:.4f}")

if p_value < alpha:
    print("✅ Statistically significant! Launch the new design.")
else:
    print("❌ Not statistically significant. Keep the old design.")

Required sample size per group: 8,143
Total sample size: 16,286

A/B TEST RESULTS:
Control CTR: 4.92%
Treatment CTR: 5.87%
Lift: 0.95%
P-value: 0.0076
✅ Statistically significant! Launch the new design.


In [8]:
def analyze_rct(control, treatment, alpha=0.05):
    """
    Complete analysis of an RCT.
    """
    n_c = len(control)
    n_t = len(treatment)
    
    mean_c = np.mean(control)
    mean_t = np.mean(treatment)
    
    std_c = np.std(control, ddof=1)
    std_t = np.std(treatment, ddof=1)
    
    # Standard error of the difference
    se_diff = np.sqrt((std_c**2 / n_c) + (std_t**2 / n_t))
    
    # Difference in means
    diff = mean_t - mean_c
    
    # Confidence interval (95%)
    z = norm.ppf(1 - alpha/2)
    ci_lower = diff - z * se_diff
    ci_upper = diff + z * se_diff
    
    # T-test
    t_stat, p_value = ttest_ind(treatment, control)
    
    print("="*50)
    print("RCT ANALYSIS RESULTS")
    print("="*50)
    print(f"Control Group: n={n_c:,}, mean={mean_c:.4f}, std={std_c:.4f}")
    print(f"Treatment Group: n={n_t:,}, mean={mean_t:.4f}, std={std_t:.4f}")
    print(f"\nAverage Treatment Effect (ATE): {diff:.4f}")
    print(f"95% Confidence Interval: [{ci_lower:.4f}, {ci_upper:.4f}]")
    print(f"T-statistic: {t_stat:.4f}")
    print(f"P-value: {p_value:.4f}")
    
    if p_value < alpha:
        print("\n✅ Statistically significant. Reject H₀.")
    else:
        print("\n❌ Not statistically significant. Fail to reject H₀.")
    
    return diff, ci_lower, ci_upper, p_value

In [9]:
diff, ci_lower, ci_upper, p_value = analyze_rct(control, treatment)

RCT ANALYSIS RESULTS
Control Group: n=8,143, mean=0.0492, std=0.2164
Treatment Group: n=8,143, mean=0.0587, std=0.2351

Average Treatment Effect (ATE): 0.0095
95% Confidence Interval: [0.0025, 0.0164]
T-statistic: 2.6706
P-value: 0.0076

✅ Statistically significant. Reject H₀.
